In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

/var/folders/04/_vl7w7bn0bg318g6c3_jh2w40000gn/T/ipykernel_15621/3450146367.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("./data/Prateek_Khare_AureetureAI_Offer_Letter.pdf")
docs = loader.load()
len(docs)

7

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

24

In [5]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [6]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [ ]:
#### Chain - Context_generate | prompt | llm | strparser

In [9]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n\n"

    return {
        "context":context,
        "question":query
    }

print(get_context("What is the subject of the offer letter?"))

{'context': "AureetureAI – Software Developer Core Team Offer Letter \nPage 1 \nAureetureAI \nAuriga of the Future \nOFFER LETTER – SOFTWARE DEVELOPER (CORE TEAM) \nDate: 04/07/2026 \nTo, \nPrateek Khare \nprateekkhare2111@gmail.com \n \nSubject: Offer of Engagement – Software Developer (Core Team Member) \nDear Prateek Khare, \nWe are pleased to offer you the position of Software Developer (Core Team Member) at \nAureetureAI, an emerging AI-driven talent discovery and hiring ecosystem.   \nYour engagement will commence with a 10-day evaluation period , during which your \ntechnical aptitude, problem -solving ability, coding skills, communication, ownership, and \ncollaboration will be assessed. Successful completion of this evaluation will lead to your \ncontinuation as a Software Developer (Core Team Member).    \nFollowing the evaluation period, you will undergo a structured onboarding programme, \nduring which you will gain a comprehensive understanding of AureetureAI's technology\

In [10]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [18]:
rag_chain = get_context | prompt | llm | StrOutputParser()

In [19]:
res = rag_chain.invoke("What is the summary of the letter? ")

In [20]:
print(res)

Based on the provided context, here is a summary of the offer letter:

* **Position & Offer:** AureetureAI (represented by Founder & CEO Rishabh Soni) has offered **Prateek Khare** the position of **Software Developer (Core Team Member)**, dated 04/07/2026.
* **Evaluation & Onboarding:** The engagement begins with a **10-day evaluation period** to assess technical aptitude, problem-solving, coding skills, communication, ownership, and collaboration. Successful completion leads to a structured onboarding program working on live production systems.
* **Growth Opportunities:** Exceptional performance may lead to leadership opportunities, increased responsibilities, future paid engagement, or equity participation at AureetureAI's discretion.
* **Governing Annexures:** The offer is governed by four annexures covering Terms & Conditions, IP Rights & Confidentiality, Operational Policies, and Joining Formalities & POSH Compliance.
* **Acceptance & Joining Details:** 
  * **Candidate:** Pratee